In [21]:
import torch
import numpy as np
import polars as pl

from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
from torch.utils.data import TensorDataset,DataLoader

In [11]:
dataset = load_breast_cancer()

X = dataset.data
y = dataset.target

print(X.shape)

(569, 30)


In [43]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = nn.Linear(30,8)
        self.fc2 = nn.Linear(8,2)

    def forward(self,x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)

        return x
        

In [48]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,
    test_size=0.2,
    stratify=y,
)

X_train = torch.tensor(X_train,dtype = torch.float32)
X_test = torch.tensor(X_test,dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train,y_train)
test_dataset = TensorDataset(X_test,y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size = 8,
    shuffle=False
)

In [55]:
model = MLP()
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.001
)

epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for X_batch,y_batch in train_loader:
        logits = model(X_batch)

        loss = criterion(logits,y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss:.4f}")

Epoch 1/20, Train Loss: 20.5159
Epoch 2/20, Train Loss: 0.9077
Epoch 3/20, Train Loss: 0.4512
Epoch 4/20, Train Loss: 0.3394
Epoch 5/20, Train Loss: 0.2536
Epoch 6/20, Train Loss: 0.2419
Epoch 7/20, Train Loss: 0.2211
Epoch 8/20, Train Loss: 0.2282
Epoch 9/20, Train Loss: 0.2288
Epoch 10/20, Train Loss: 0.2362
Epoch 11/20, Train Loss: 0.2179
Epoch 12/20, Train Loss: 0.2224
Epoch 13/20, Train Loss: 0.2005
Epoch 14/20, Train Loss: 0.1999
Epoch 15/20, Train Loss: 0.2184
Epoch 16/20, Train Loss: 0.2047
Epoch 17/20, Train Loss: 0.2429
Epoch 18/20, Train Loss: 0.2387
Epoch 19/20, Train Loss: 0.2110
Epoch 20/20, Train Loss: 0.2130


In [53]:
model.eval()

correct = 0
total = 0
test_loss_sum = 0

with torch.no_grad():
    for X_batch,y_batch in test_loader:

        logits = model(X_batch)
        loss = criterion(logits,y_batch)

        test_loss_sum += loss.item() * X_batch.size(0)
        predictions = logits.argmax(dim=1)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

    test_loss = test_loss_sum / total
    test_accuracy = correct / total

print(test_loss)
print(test_accuracy)

0.20659126609302403
0.9035087719298246
